In [1]:
from kafka import KafkaConsumer
from kafka import TopicPartition
from datetime import datetime
import pandas as pd
from cassandra.cluster import Cluster
import json
import pyspark.sql 
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import HashingTF, Tokenizer
from pyspark.ml import Pipeline, PipelineModel
from pyspark.sql import SQLContext
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
import pandas as pd
import time
from cassandra.cluster import Cluster
import numpy as np
from pyspark.sql.types import *

In [2]:
# #------------------------------- SPARK CONTEXT -------------------------------
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CC_Fraud")
    .getOrCreate()
)

sc = spark.sparkContext


26/06/28 07:58:44 WARN Utils: Your hostname, Aryas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.58 instead (on interface en0)
26/06/28 07:58:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 07:58:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
def MachineLearningModelLoading():
    return PipelineModel.load("../models/fraud_pipeline")

newmodel = MachineLearningModelLoading()

In [4]:
# feature_columns = [col for col in df.columns if col != "is_fraud"]
# assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

In [5]:
def predicting_status(data_all,newmodel):
	select_cols = ['amt', 'lat', 'long', 'city_pop', 'unix_time','merch_lat', 'merch_long','merchant_label','category_label', 'gender_label', 'job_label'] #all numeric
	data = data_all.select(select_cols)
	pred1 = newmodel.transform(data)
	return pred1.select("prediction")

# df = spark.read.json(sc.parallelize([dd]))
# res = predicting_status(df, newmodel)

In [6]:
cluster = Cluster()
session = cluster.connect()

session.execute("CREATE KEYSPACE IF NOT EXISTS bigdata WITH replication = {'class':'SimpleStrategy', 'replication_factor' : 1}")
session.set_keyspace("bigdata")

#session.execute("DROP TABLE testing")
# session.execute("CREATE TABLE IF NOT EXISTS testing (txn_id varchar PRIMARY KEY,cc_no bigint ,txn_time varchar, year smallint, month smallint, day smallint, hour smallint, minute smallint,amount double, cc_provider int, merchant varchar,merchant_index smallint, country_name varchar, country_code varchar,country_index smallint, age int, marital_status varchar,marital_status_index smallint,gender varchar,gender_index smallint,loan varchar,loan_index smallint, status varchar,status_index smallint)")


In [7]:
topic_name = 'transaction_data'
keyspace = 'bigdata'
table = 'transaction_data'

In [ ]:
consumer = KafkaConsumer(
    topic_name,
    group_id=None,
    bootstrap_servers=["localhost:9092"]
)

for msg in consumer:
    try:
        print("msg", msg)

        msg = msg.value.decode("utf-8")
        output = json.loads(msg)

        df = spark.read.json(sc.parallelize([msg]))
        status = predicting_status(df, newmodel)

        status = int(status.collect()[0][0])

        output["is_fraud_prediction"] = status
        output["inserted_at"] = int(time.time() * 1000)

        insert_query = f"INSERT INTO {keyspace}.{table} JSON %s"
        session.execute(insert_query, [json.dumps(output)])

    except Exception as e:
        print("Error in transaction:", e)

msg ConsumerRecord(topic='transaction_data', partition=0, leader_epoch=2, offset=403, timestamp=1782652298959, timestamp_type=0, key=b'null', value=b'{"trans_date_trans_time": "2020-08-18 23:20:37", "cc_num": 4586810168620942, "merchant": "fraud_Lowe, Dietrich and Erdman", "category": "kids_pets", "amt": 6.58, "first": "Michelle", "last": "Gregory", "gender": "F", "street": "6983 Carrillo Isle", "city": "Edisto Island", "state": "SC", "zip": 29438, "lat": 32.5486, "long": -80.307, "city_pop": 2408, "job": "Sales professional, IT", "dob": "1997-07-05", "trans_num": "f8e8a0e616d7e7cbeaa239b4d5dfb173", "unix_time": 1376868037, "merch_lat": 32.005327, "merch_long": -80.33445400000001, "is_fraud": 0, "merchant_label": 396, "category_label": 7, "gender_label": 0, "job_label": 406}', headers=[], checksum=None, serialized_key_size=4, serialized_value_size=635, serialized_header_size=-1)


26/06/28 08:11:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


msg ConsumerRecord(topic='transaction_data', partition=0, leader_epoch=2, offset=404, timestamp=1782652298959, timestamp_type=0, key=b'null', value=b'{"trans_date_trans_time": "2020-10-24 15:53:49", "cc_num": 675990301623, "merchant": "fraud_Botsford PLC", "category": "home", "amt": 15.3, "first": "Amanda", "last": "Spencer", "gender": "F", "street": "6682 Green Forks", "city": "Ogdensburg", "state": "NJ", "zip": 7439, "lat": 41.0767, "long": -74.5982, "city_pop": 2456, "job": "Senior tax professional/tax inspector", "dob": "1994-03-13", "trans_num": "f25106044506b009d3904fd5eb43db70", "unix_time": 1382630029, "merch_lat": 41.5018, "merch_long": -74.888103, "is_fraud": 0, "merchant_label": 68, "category_label": 6, "gender_label": 0, "job_label": 421}', headers=[], checksum=None, serialized_key_size=4, serialized_value_size=610, serialized_header_size=-1)
msg ConsumerRecord(topic='transaction_data', partition=0, leader_epoch=2, offset=405, timestamp=1782652298959, timestamp_type=0, key=

In [ ]:
consumer.close()
spark.stop()

In [ ]:
# insert_query = f"INSERT INTO {keyspace}.{table} JSON '{json.dumps(output)}';"
# session.execute(insert_query)